# Project 12 — Local Policy Assistant
## Search HR/IT/Company Policies with Citations

**Stack:** Ollama · LangChain · ChromaDB · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb

## Step 1 — Setup

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

llm = ChatOllama(model="qwen3:8b", temperature=0.1)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

## Step 2 — Create Sample Policy Documents

In [ ]:
from pathlib import Path

policies_dir = Path("sample_data/policies"); policies_dir.mkdir(parents=True, exist_ok=True)

policies = {
    "remote_work_policy.txt": """Remote Work Policy — Effective Jan 2025
Section 1: Eligibility
All full-time employees who have completed 90 days are eligible for remote work.
Contractors and part-time employees require manager approval.

Section 2: Equipment
The company provides a laptop, monitor, and $500 home office stipend.
Equipment must be returned within 14 days of separation.

Section 3: Working Hours
Core hours are 10am-3pm in your local timezone. Employees must be
available on Slack during core hours. Total weekly hours: 40.

Section 4: Security
All work must be done on company-approved devices. Use the VPN for
accessing internal systems. Do not use public WiFi without VPN.""",

    "pto_policy.txt": """Paid Time Off Policy — Effective Jan 2025
Section 1: Accrual
Full-time employees accrue 1.67 days per month (20 days/year).
Maximum accrual cap: 30 days. Unused PTO does not roll over beyond the cap.

Section 2: Requesting PTO
Submit requests via HR Portal at least 5 business days in advance.
Requests of 5+ consecutive days require VP approval.

Section 3: Holidays
The company observes 10 federal holidays plus 2 floating holidays.
Floating holidays must be used within the calendar year.

Section 4: Sick Leave
Employees receive 10 sick days per year, separate from PTO.
A doctor's note is required for absences exceeding 3 consecutive days.""",

    "expense_policy.txt": """Expense Reimbursement Policy — Effective Jan 2025
Section 1: Eligible Expenses
Business travel, client meals, conference fees, and work-related software.

Section 2: Limits
Meals: $50/person for client dinners, $25 for solo meals.
Hotels: Up to $250/night in major metros, $175 elsewhere.
Flights: Economy class for domestic, business class for 6+ hour flights.

Section 3: Submission
Submit expenses within 30 days via Concur. Receipts required for all
expenses over $25. Late submissions may be denied.

Section 4: Approval
Expenses under $500: Manager approval.
Expenses $500-$5000: Director approval.
Expenses over $5000: VP approval.""",
}

for fname, content in policies.items():
    (policies_dir / fname).write_text(content, encoding="utf-8")
print(f"Created {len(policies)} policy documents")

## Step 3 — Index Policies with Metadata

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

loader = DirectoryLoader("sample_data/policies", glob="*.txt", loader_cls=TextLoader,
                         loader_kwargs={"encoding": "utf-8"})
docs = loader.load()

# Enrich metadata
for doc in docs:
    fname = Path(doc.metadata["source"]).name
    doc.metadata["policy_name"] = fname.replace("_policy.txt", "").replace("_", " ").title()
    doc.metadata["type"] = "company_policy"

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks = splitter.split_documents(docs)

vectorstore = Chroma.from_documents(chunks, embeddings,
    persist_directory="sample_data/policy_chroma", collection_name="policies")

print(f"Indexed {len(chunks)} policy chunks from {len(docs)} documents")

## Step 4 — Policy Q&A with Section Citations

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

policy_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are the company's HR policy assistant. Answer questions
using ONLY the policy documents provided. Always cite the specific
policy name and section number.

Format: Answer the question, then cite as "Source: [Policy Name], Section X"

If the answer is not in the policies, say "This is not covered in our
current policies. Please contact HR at hr@company.com."

Policy excerpts:
{context}

Employee Question: {question}

Answer (with citations):"""
)

qa = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": policy_prompt},
)

questions = [
    "How many PTO days do I get per year?",
    "What's the hotel reimbursement limit in New York?",
    "Am I eligible for remote work as a contractor?",
    "Do I need approval for a $600 expense?",
    "What happens to unused PTO?",
    "Can I use public WiFi when working remotely?",
]

for q in questions:
    print(f"\nQ: {q}")
    result = qa.invoke({"query": q})
    print(f"A: {result['result']}")
    policies_cited = set(s.metadata.get('policy_name', '?') for s in result['source_documents'])
    print(f"Policies referenced: {policies_cited}")

## What You Learned
- **Multi-document policy indexing** with metadata enrichment
- **Citation-aware retrieval** with specific section references
- **Graceful fallback** for uncovered topics